# 02e · Y1: YOLOv8 (Selva Kumar et al., ICETCS 2024)

The paper trains YOLOv8 **instance segmentation** on polygon-annotated leaves (Rust, Miner, Phoma) and reads the
label off the rendered output with EasyOCR. Our dataset has class labels only (no polygons), so we use the
classification variant **YOLOv8s-cls** with the same backbone family, and say so in the paper.

| Paper | Here |
|---|---|
| YOLOv8 (ultralytics), pretrained | `yolov8s-cls.pt` (ImageNet-pretrained), ultralytics trainer |
| augmentation: flipping, rotation, brightness | horizontal + vertical flip (0.5 each; with both flips 4 of the 8 rotations/flips are covered), brightness (hsv_v 0.4); ultralytics' other default augmentations off |
| epochs not stated | 60 epochs over the ~2,500 byte-unique training images = ~150k samples, the same training budget as V2 |
| images | our group-aware folds; training folder = the byte-unique training images (hard links, no copying) |

Ultralytics picks its best epoch on the validation images (top-1 accuracy). Time: roughly 6-10 min per fold.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import os
import shutil
import time

import numpy as np
import pandas as pd
import torch

from common import config, data, env
from common.cv import folds_to_run, preds_path
from common.metrics import classification_metrics
from common.train import measure_cost, save_preds

config.ensure_dirs()
device = env.require_cuda()
env.seed_everything()
uniq, splits = data.load_unique(), data.load_splits()
YDATA = config.RUNS_DIR / "Y1_data"
YRUNS = config.RUNS_DIR / "Y1"
EPOCHS = 1 if config.SMOKE else 60
WEIGHTS = config.ROOT / "_cache" / "yolov8s-cls.pt"   # downloaded once by ultralytics

In [ ]:
def link(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)          # hard link: no extra disk space (same drive)
    except OSError:
        shutil.copy2(src, dst)


def build_fold_dir(k: int, idx: dict) -> Path:
    root = YDATA / f"fold{k}"
    marker = root / "done.json"
    if marker.exists():
        return root
    if root.exists():
        shutil.rmtree(root)
    train_files = uniq.iloc[idx["train"]]       # byte-unique training images (same pool as every other model)
    for p, c in zip(train_files["path"], train_files["cls"]):
        link(config.DATA_DIR / p, root / "train" / c / Path(p).name)
    for p, c in zip(uniq["path"].to_numpy()[idx["val"]], uniq["cls"].to_numpy()[idx["val"]]):
        link(config.DATA_DIR / p, root / "val" / c / Path(p).name)
    marker.write_text(json.dumps({"train_files": int(len(train_files)), "val_files": int(len(idx["val"]))}))
    return root

In [ ]:
from ultralytics import YOLO

if not WEIGHTS.exists():
    WEIGHTS.parent.mkdir(exist_ok=True)
    cwd = os.getcwd()
    os.chdir(WEIGHTS.parent)          # ultralytics downloads named weights into the working directory
    try:
        YOLO("yolov8s-cls.pt")
    finally:
        os.chdir(cwd)


def yolo_logits(model, paths):
    """Class probabilities in OUR class order, returned as log-probabilities (softmax(log p) = p)."""
    names = model.names                       # {index: folder name}
    order = [list(names.values()).index(c) for c in config.CLASSES]
    out = []
    for s in range(0, len(paths), 64):
        res = model.predict([str(config.DATA_DIR / p) for p in paths[s:s + 64]], imgsz=224, device=0,
                            verbose=False)
        out += [r.probs.data.float().cpu().numpy()[order] for r in res]
    return np.log(np.clip(np.stack(out), 1e-12, 1.0))


per_fold = {}
for k in folds_to_run():
    idx = data.fold_indices(uniq, splits, k)
    run_dir = YRUNS / f"fold{k}"
    best = run_dir / "weights" / "best.pt"
    if not (best.exists() and preds_path("Y1", k, "test").exists()):
        root = build_fold_dir(k, idx)
        t0 = time.time()
        model = YOLO(str(WEIGHTS))
        model.train(data=str(root), epochs=EPOCHS, imgsz=224, batch=64, device=0, workers=4, seed=config.SEED,
                    deterministic=False, amp=True, project=str(YRUNS), name=f"fold{k}", exist_ok=True,
                    pretrained=True, plots=False, verbose=False, val=True,
                    fliplr=0.5, flipud=0.5, hsv_v=0.4, hsv_h=0.0, hsv_s=0.0, auto_augment=None, erasing=0.0,
                    scale=0.0, degrees=0.0, translate=0.0)
        minutes = (time.time() - t0) / 60
        (run_dir / "time.json").write_text(json.dumps({"train_minutes": minutes}))
    minutes = json.loads((run_dir / "time.json").read_text())["train_minutes"]
    model = YOLO(str(best))
    for split in ("val", "test"):
        paths = uniq["path"].to_numpy()[idx[split]]
        lg = yolo_logits(model, list(paths))
        save_preds(preds_path("Y1", k, split), uniq, lg, uniq["label"].to_numpy()[idx[split]], idx[split])
    p = pd.read_csv(preds_path("Y1", k, "test"))
    tm = classification_metrics(p["y_true"].to_numpy(), p[[f"logit_{c}" for c in range(5)]].to_numpy().argmax(1))
    per_fold[k] = {"train_minutes": minutes, "test": tm}
    print(f"[Y1] fold {k}: {minutes:.1f} min | TEST acc {tm['accuracy']:.4f} macro-F1 {tm['macro_f1']:.4f}")
    torch.cuda.empty_cache()

In [ ]:
env.write_json(config.CNN_RESULTS / "Y1_summary.json",
               {"model": "Y1", "config": {"weights": "yolov8s-cls.pt", "epochs": EPOCHS, "imgsz": 224, "batch": 64,
                                          "augment": "fliplr 0.5, flipud 0.5, hsv_v 0.4"},
                "folds": per_fold, "meta": env.run_meta()})
cost = measure_cost(YOLO(str(best)).model, 224, "Y1")
env.write_json(config.CNN_RESULTS / "Y1_cost.json", cost)
print(cost)
pd.DataFrame({k: {"minutes": round(v["train_minutes"], 1), "test_acc": v["test"]["accuracy"],
                  "test_macro_f1": v["test"]["macro_f1"]} for k, v in per_fold.items()}).T